In [ ]:
import os
import time
import warnings
from datetime import datetime
from zoneinfo import ZoneInfo
import ipywidgets as widgets
import pandas as pd
import requests
import xml.etree.ElementTree as ET
from IPython.display import display, clear_output

warnings.filterwarnings('ignore')

from getpass import getpass
API_AUTH_KEY = os.environ.get("WORK24_API_KEY") or getpass("고용24 API 인증키: ")

# KST 시간대 설정
def get_kst_now():
    try:
        return datetime.now(ZoneInfo("Asia/Seoul"))
    except ImportError:
        try:
            import pytz
            kst = pytz.timezone('Asia/Seoul')
            return datetime.now(kst)
        except ImportError:
            import timezone
            kst = timezone(timedelta(hours=9))
            return datetime.now(kst)

# --- 데이터 수집 클래스 정의 ---
class Work24APICollector:
    """고용24 API를 활용한 채용정보 데이터 수집기"""

    def __init__(self, auth_key: str):
        self.auth_key = auth_key
        self.base_url = "https://www.work24.go.kr/cm/openApi/call/wk/callOpenApiSvcInfo210L01.do"
        self.session = requests.Session()

    def build_params(self, **kwargs) -> dict:
        """API 요청 파라미터 구성"""
        params = {
            'authKey': self.auth_key,
            'callTp': 'L',
            'returnType': 'XML',
            'startPage': kwargs.get('startPage', 1),
            'display': kwargs.get('display', 100)
        }
        if 'regDate' in kwargs and kwargs['regDate']:
            params['regDate'] = kwargs['regDate']
        return params

    def parse_xml_response(self, xml_content: str) -> dict:
        """XML 응답 파싱"""
        try:
            root = ET.fromstring(xml_content)
            if root.find('.//total') is None:
                error_message = root.find('.//message').text if root.find('.//message') is not None else "알 수 없는 오류"
                print(f" API 오류 응답: {error_message}")
                return {'total': 0, 'wanted_list': []}

            total = int(root.find('.//total').text)
            wanted_list = [
                {child.tag: child.text or '' for child in wanted}
                for wanted in root.findall('.//wanted')
            ]
            return {'total': total, 'wanted_list': wanted_list}
        except ET.ParseError as e:
            print(f" XML 파싱 오류: {e}\n응답 내용: {xml_content[:300]}")
            return {'total': 0, 'wanted_list': []}

    def fetch_data(self, **kwargs) -> dict:
        """단일 페이지 데이터 조회"""
        params = self.build_params(**kwargs)
        try:
            response = self.session.get(self.base_url, params=params, timeout=30)
            response.raise_for_status()
            return self.parse_xml_response(response.text)
        except requests.exceptions.RequestException as e:
            print(f" API 요청 오류: {e}")
            return {'total': 0, 'wanted_list': []}

    def collect_all_pages(self, **kwargs) -> list:
        """모든 페이지 데이터 수집"""
        all_data = []
        current_page = 1
        display = 100
        max_pages_to_fetch = 50  # 과도한 요청 방지를 위해 50페이지로 제한

        with output:
            clear_output(wait=True)
            print("데이터 수집을 시작해...")
            while current_page <= max_pages_to_fetch:
                print(f"페이지 {current_page} 데이터 수집 중...")
                result = self.fetch_data(startPage=current_page, display=display, **kwargs)

                if not result['wanted_list']:
                    print(" 더 이상 수집할 데이터가 없음.")
                    break

                all_data.extend(result['wanted_list'])
                print(f"   > {len(result['wanted_list'])}건 수집 완료 (누적: {len(all_data)}건)")

                if len(result['wanted_list']) < display:
                    print(" 마지막 페이지에 도달.")
                    break
                current_page += 1
                time.sleep(0.5)  # API 서버 부하 감소를 위한 딜레이

            print("-" * 30)
            print(f"전체 데이터 수집 완료: 총 {len(all_data)}건")
        return all_data


def align_dataframes(new_df: pd.DataFrame, existing_df: pd.DataFrame) -> tuple:
    """두 DataFrame의 컬럼을 통일하여 반환"""
    all_columns = list(set(new_df.columns) | set(existing_df.columns))
    all_columns.sort()

    # 각 DataFrame에 없는 컬럼을 빈 값으로 추가
    for col in all_columns:
        if col not in new_df.columns:
            new_df[col] = ''
        if col not in existing_df.columns:
            existing_df[col] = ''

    # 컬럼 순서 통일
    new_df = new_df[all_columns]
    existing_df = existing_df[all_columns]

    return new_df, existing_df


# --- 위젯 이벤트 핸들러 정의 ---

collector = Work24APICollector(API_AUTH_KEY)

def test_api_connection(b):
    """API 연결 테스트 함수"""
    with output:
        clear_output(wait=True)
        print(" API 연결 테스트를 시작...")
        if not collector.auth_key or collector.auth_key.isspace():
            print(" API 인증키 설정 'API_AUTH_KEY' 변수에 실제 키를 입력할것.")
            return

        result = collector.fetch_data(display=1)  # 1건만 테스트
        if result['wanted_list']:
            print(" API 연결 성공! 정상적으로 데이터를 수신.")
            print(f" 수신된 컬럼: {list(result['wanted_list'][0].keys())}")
        else:
            print(" API 연결은 되었으나 데이터 수신에 실패, 인증키 확인")

def run_collection(b):
    """데이터 수집 실행 및 중복 제거하여 master 파일에 저장하는 함수"""
    save_path = '/content/drive/My Drive/Colab Notebooks/work24/'
    master_file_path = os.path.join(save_path, 'worknet_jobs_master.csv')

    # 폴더가 없으면 생성
    if not os.path.exists(save_path):
        os.makedirs(save_path)
        print(f"'{save_path}' 폴더를 생성")

    # 데이터 수집 실행
    selected_reg_date = regdate_dropdown.value
    job_data = collector.collect_all_pages(regDate=selected_reg_date)

    with output:
        if not job_data:
            print(" 수집된 데이터가 없음")
            return

        new_df = pd.DataFrame(job_data)
        print(f"신규 수집 데이터: {len(new_df)}건")
        print(f"신규 데이터 컬럼 ({len(new_df.columns)}개): {list(new_df.columns)}")

        if os.path.exists(master_file_path):
            print(f"기존 master 파일을 확인: {master_file_path}")
            try:
                # CSV 읽기 시 파싱 에러 방지 옵션 추가
                existing_df = pd.read_csv(
                    master_file_path,
                    encoding='utf-8-sig',
                    on_bad_lines='skip',
                    quoting=1,
                    engine='python'
                )

                print(f"기존 데이터 컬럼 ({len(existing_df.columns)}개): {list(existing_df.columns)}")

                # 컬럼 수가 다른 경우 컬럼 통일 처리
                if len(new_df.columns) != len(existing_df.columns):
                    print(f" 컬럼 수 불일치 감지 - 컬럼을 통일")
                    new_df, existing_df = align_dataframes(new_df, existing_df)
                    print(f" 컬럼 통일 완료 ({len(new_df.columns)}개): {list(new_df.columns)}")

                # 중복 제거 로직
                if 'wantedAuthNo' in new_df.columns and 'wantedAuthNo' in existing_df.columns:
                    existing_ids = set(existing_df['wantedAuthNo'].astype(str))
                    new_jobs_df = new_df[~new_df['wantedAuthNo'].astype(str).isin(existing_ids)]
                else:
                    print(" wantedAuthNo 컬럼이 없어 중복 체크 불가, 전체 데이터를 추가")
                    new_jobs_df = new_df

                if not new_jobs_df.empty:
                    print(f" {len(new_jobs_df)}건의 새로운 채용공고를 master 파일에 추가")

                    # 기존 파일에 새 데이터 추가 (컬럼 통일된 상태로)
                    combined_df = pd.concat([existing_df, new_jobs_df], ignore_index=True)
                    combined_df.to_csv(master_file_path, index=False, encoding='utf-8-sig')
                    print(f" Master 파일 업데이트 완료 (총 {len(combined_df)}건)")
                else:
                    print("새로운 채용공고가 없음, Master 파일은 최신 상태")

            except pd.errors.ParserError as e:
                print(f"기존 master 파일 읽기 실패: {e}")
                print("기존 파일을 백업하고 새로운 master 파일을 생성")

                # KST 시간으로 백업 파일명 생성
                kst_now = get_kst_now()
                backup_path = master_file_path.replace('.csv', f'_backup_{kst_now.strftime("%Y%m%d_%H%M%S")}.csv')
                if os.path.exists(master_file_path):
                    os.rename(master_file_path, backup_path)
                    print(f" 기존 파일 백업: {backup_path}")

                # 새로운 master 파일 생성
                new_df.to_csv(master_file_path, index=False, encoding='utf-8-sig')
                print(f" 새로운 master 파일 생성: {master_file_path}")

            except Exception as e:
                print(f"예상치 못한 오류: {e}")
                print("새로운 master 파일을 생성.")
                new_df.to_csv(master_file_path, index=False, encoding='utf-8-sig')
        else:
            print(f"새로운 master 파일 생성: {master_file_path}")
            new_df.to_csv(master_file_path, index=False, encoding='utf-8-sig')

        # --- 날짜별 백업 파일 저장 로직 ---
        kst_now = get_kst_now()
        timestamp_filename = f"worknet_jobs_{kst_now.strftime('%Y%m%d')}.csv"
        timestamp_full_path = os.path.join(save_path, timestamp_filename)
        new_df.to_csv(timestamp_full_path, index=False, encoding='utf-8-sig')
        print(f" 백업 파일 저장 완료: {timestamp_full_path} (KST: {kst_now.strftime('%Y-%m-%d %H:%M:%S')})")

        print("\n 신규 수집 데이터 미리보기 (상위 5개)")
        display(new_df.head())


# --- 위젯 UI 표시 ---
regdate_dropdown = widgets.Dropdown(
    options=[('오늘 등록', 'D-0'), ('3일 이내', 'D-3'), ('1주 이내', 'W-1'), ('2주 이내', 'W-2'), ('1개월 이내', 'M-1')],
    value='D-0',
    description='등록일 조건:',
    style={'description_width': 'initial'}
)

# 버튼 위젯
test_button = widgets.Button(description='API 연결 테스트', button_style='info', icon='wifi')
run_button = widgets.Button(description='데이터 수집 실행', button_style='success', icon='play')

# 출력 영역 위젯
output = widgets.Output()

# 사용 가이드 위젯
info_html = widgets.HTML(
    value="""
    <h3> 고용24 채용정보 수집기 </h3>
    <p><b>사용방법</b></p>
    <p><b> 주의사항: 웹사이트에서 1일 기준 10만 PV(Page View), 1만 쿼리 미만으로 사용할것!</b></p>
    <ol>
        <li><b>연결 테스트:</b> 'API 연결 테스트' 버튼으로 상태를 확인</li>
        <li><b>조건 선택:</b> 원하는 '등록일 조건'을 선택</li>
        <li><b>수집 실행:</b> '데이터 수집 실행' 버튼을 클릭</li>
    </ol>
    <p><small>※ 모든 시간은 한국 표준시(KST) 기준으로 저장됩니다.</small></p>
    """
)

# 이벤트 핸들러 연결
test_button.on_click(test_api_connection)
run_button.on_click(run_collection)

# 모든 위젯을 VBox로 묶어서 표시
widget_box = widgets.VBox([
    info_html,
    regdate_dropdown,
    widgets.HBox([test_button, run_button]),
    widgets.HTML("<hr>"),
    output
])

display(widget_box)